# Hafta 6 — Bitirme Projesi: Akademik Asistan Ajanı

> [`src/academic_assistant/`](../src/academic_assistant/) **için kaynak not defteri.**

Bu entegrasyon haftasıdır. Hafta 1–5'te kurulan tüm bileşenleri, bir araştırma konusu verildiğinde yapılandırılmış bir Markdown literatür taraması üreten tek bir otonom sistemde birleştiriyoruz.

## Hat

```
        konu
          │
          ▼
  ┌────────────────┐   arXiv API
  │ arxiv_ingest   │◀───────────────  (Hafta 6)
  └───────┬────────┘
          │ Paper nesneleri
          ▼
  ┌────────────────┐
  │  PaperStore    │  SQLite, MCP ile salt okunur sunulur   (Hafta 1)
  └───────┬────────┘
          │
          ▼
  ┌────────────────┐
  │ HierarchicalRAG│  ebeveyn-çocuk parçalar + Chroma       (Hafta 2)
  └───────┬────────┘
          │ adaylar
          ▼
  ┌────────────────┐
  │ CrossEncoder   │  kesinlik için yeniden sıralama        (Hafta 3)
  │  Reranker      │
  └───────┬────────┘
          │ bölüm başına ilk k
          ▼
  ┌────────────────┐
  │  LLM sentezi   │  ReAct tarzı akıl yürütme              (Hafta 4–5)
  └───────┬────────┘
          │
          ▼
   Markdown inceleme
```

## Öğrenme hedefleri

1. Bağımsız olarak test edilmiş modülleri uçtan uca otonom bir sistemde birleştirmek.
2. Alınan makaleleri kalıcı hâle getirmek ve sunmak için MCP SQLite kaynak katmanını kullanmak.
3. LLM'i temellendirilmiş iddialarla sınırlayan "önce ana hat, sonra sentez" üretim stratejisini yönetmek.
4. Parçalarınkinden farklı olan *birleşik* sistemin hata türleri üzerine akıl yürütmek.

> **Not:** Derlem ve gömme modeli İngilizce olduğundan örnek makaleler ve bölüm başlıkları (erişim sorguları) İngilizce bırakılmıştır; sentezlenen rapor metni Türkçedir.

## 1. Veri alma: arXiv → yapılandırılmış Paper nesneleri

[`src/academic_assistant/arxiv_ingest.py`](../src/academic_assistant/arxiv_ingest.py) arXiv API'sini sarmalar ve tipli `Paper` nesneleri üretir. Not defterinin çevrimdışı tekrarlanabilir olması için canlı çağrı yerine küçük bir test verisi kullanıyoruz; üretimde `fetch_papers(topic, max_results=25)` çağırırsınız.

In [ ]:
from src.academic_assistant.arxiv_ingest import Paper

# Canlı bir arXiv çekiminin yerini tutan küçük, çevrimdışı bir test verisi.
fixture = [
    Paper("2212.10496", "Precise Zero-Shot Dense Retrieval without Relevance Labels",
          ["Luyu Gao", "Xueguang Ma", "Jimmy Lin", "Jamie Callan"],
          "We propose HyDE, which generates a hypothetical document via an instruction-following "
          "language model and embeds it for retrieval, outperforming dense retrievers without labels.",
          "2022-12-20", "https://arxiv.org/pdf/2212.10496", ["cs.IR"]),
    Paper("2210.03629", "ReAct: Synergizing Reasoning and Acting in Language Models",
          ["Shunyu Yao", "Jeffrey Zhao", "Dian Yu", "Nan Du"],
          "ReAct interleaves reasoning traces and task-specific actions, allowing the model to "
          "induce, track, and update action plans while interfacing with external tools.",
          "2022-10-06", "https://arxiv.org/pdf/2210.03629", ["cs.CL"]),
    Paper("1908.10084", "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks",
          ["Nils Reimers", "Iryna Gurevych"],
          "Sentence-BERT modifies BERT with siamese and triplet networks to derive semantically "
          "meaningful sentence embeddings comparable with cosine similarity.",
          "2019-08-27", "https://arxiv.org/pdf/1908.10084", ["cs.CL"]),
]
for p in fixture:
    print(f"{p.arxiv_id}: {p.title[:60]}")


## 2. Kalıcılık: MCP destekli SQLite deposu

`PaperStore` makaleleri SQLite'a yazar. Aynı veritabanı, `SQLiteResources` (Hafta 1) aracılığıyla herhangi bir MCP istemcisine *salt okunur* olarak sunulur — böylece MCP sunucumuza bağlı bir Claude Desktop oturumu, alınan makalelere bir kaynak olarak göz atabilir.

In [ ]:
import tempfile
from pathlib import Path
from src.academic_assistant.store import PaperStore

db_path = Path(tempfile.mktemp(suffix=".sqlite"))
store = PaperStore(db_path)
store.upsert_many(fixture)

print("depodaki satır sayısı:", len(store.all_papers()))
for row in store.all_papers():
    print(" ", row["arxiv_id"], "-", row["title"][:50])


In [ ]:
# MCP kaynak katmanı (Hafta 1) aracılığıyla salt okunur olarak sun.
from src.mcp_core.resources import SQLiteResources
store.close()   # salt okunur bir tanıtıcı açmadan önce yazıcıyı kapat

mcp_resources = SQLiteResources(db_path)
print("Duyurulan MCP kaynakları:")
for r in mcp_resources.list_resources():
    print(" ", r["uri"], f"({r['mimeType']})")
print("\nMCP ile okundu:")
print(mcp_resources.read("sqlite://papers")[0]["title"])


## 3. İndeksleme ve erişim (Hafta 2–3)

Özetler hiyerarşik olarak parçalanır, gömülür ve Chroma'da saklanır. Ardından örnek bir bölüm sorgusu için erişim yapıp yeniden sıralıyoruz.

In [ ]:
from src.rag_engine.chunking import HierarchicalChunker
from src.rag_engine.embeddings import SentenceEncoder
from src.rag_engine.pipeline import HierarchicalRAG
from src.rag_engine.vector_stores import build_store
from src.reranking.cross_encoder import CrossEncoderReranker
from src.reranking.pipeline import RerankingPipeline

persist = tempfile.mkdtemp()
rag = HierarchicalRAG(
    chunker=HierarchicalChunker(parent_chars=600, child_chars=250),
    encoder=SentenceEncoder("BAAI/bge-small-en-v1.5"),
    child_store=build_store("chroma", collection="capstone", persist_dir=persist),
    parent_lookup={},
)
for p in fixture:
    rag.ingest_document(
        f"# {p.title}\n\n{p.abstract}",
        metadata={"arxiv_id": p.arxiv_id, "title": p.title, "published": p.published},
    )

reranker = RerankingPipeline(reranker=CrossEncoderReranker(), candidate_k=5)

section = "approaches to zero-shot dense retrieval"
candidates = rag.retrieve(section, k=3)
top = reranker.rerank(section, candidates, k=2)
print(f"Bölüm: {section}\n")
for d in top:
    print(f"  [{d.cross_encoder_score:+.3f}] {d.metadata['title'][:60]}")


## 4. "Önce ana hat, sonra sentez" stratejisi

Saf bir "bütün bu makaleleri özetle" istemi biçimsiz bir metin duvarı üretir. Bunun yerine asistan:

1. LLM'den bir **bölüm ana hattı** ister (tutarlı bir hikâye anlatan 4–6 başlık).
2. Her bölüm için en ilgili alıntıları getirir + yeniden sıralar.
3. LLM'den, **yalnızca** getirilen alıntılara dayanarak *o tek bölümü* yazmasını ister.
4. Yapılandırılmış `Paper` üst verilerinden oluşturulan bir Kaynakça bölümü ekler.

Çıktıyı temellendirilmiş tutan bu yapıdır: sentez adımına küçük bir yeniden sıralanmış alıntı kümesi verip uydurma atıfları yasaklayarak LLM'i destekleyebileceği iddialarla sınırlarız. Uygulamanın tamamı [`src/academic_assistant/pipeline.py`](../src/academic_assistant/pipeline.py) dosyasındadır.

In [ ]:
from src.academic_assistant.pipeline import AcademicAssistant, AssistantConfig

# Not defteri çevrimdışı çalışsın diye senaryolu LLM. Üretimde build_client(...) kullanılır.
class FakeLLM:
    def __init__(self, scripted): self.scripted = list(scripted)
    def complete(self, messages, **kw):
        return self.scripted.pop(0) if self.scripted else "(senaryolu yanıt kalmadı)"

outline = "Dense Retrieval Foundations\nZero-Shot Retrieval with HyDE\nReasoning and Acting Agents"
section_text = (
    "## Bölüm\n\nSon çalışmalar, cümle gömmelerini yoğun erişimin omurgası olarak "
    "konumlandırır [Reimers ve Gurevych, 2019]. Bunun üzerine inşa edilen HyDE, üretilmiş "
    "varsayımsal bir belgeyi gömerek ilgililik etiketlerine olan ihtiyacı ortadan kaldırır [Gao vd., 2022]."
)
fake = FakeLLM([outline] + [section_text] * 3)

assistant = AcademicAssistant(llm=fake)
# Not defterinde canlı bir arXiv çekiminden kaçınmak için iç yöntemleri doğrudan çağırıyoruz.
sections = assistant._build_outline("agentic retrieval", 3)
print("ANA HAT:")
for s in sections:
    print("  -", s)


In [ ]:
# Yeniden sıralanmış alıntılardan bir bölüm sentezle.
md_section = assistant._synthesize_section(sections[1], top)
print(md_section)


In [ ]:
# Kaynakça bölümü yapılandırılmış üst verilerden deterministik olarak oluşturulur —
# LLM işin içinde olmadığından atıflar uydurulamaz.
print(assistant._references_section(fixture))


## 5. Her şeyi çalıştırmak

Üretimde hattın tamamı komut satırından çalışır:

```bash
python -m src.academic_assistant.cli \
    --topic "self-rewarding language models" \
    --max-papers 25 \
    --sections 5 \
    --output reports/self_rewarding_lm_review.md
```

Bu tek komut: arXiv'den 25 makale çeker, bunları MCP ile sunulan SQLite deposunda kalıcı hâle getirir, özetlerini hiyerarşik parçalamayla indeksler, 5 bölümlük bir ana hat oluşturur, her bölüm için alıntıları getirip cross-encoder ile yeniden sıralar, her bölümü temellendirilmiş atıflarla sentezler ve eksiksiz bir Markdown literatür taraması yazar.

## 6. *Birleşik* sistemin hata türleri

Birleştirme, parçaların hiçbirinin tek başına göstermediği hata türleri ortaya çıkarır:

| Hata | Nerede ortaya çıkar | Önlem |
|---------|------------------|-----------|
| Bir bölüm için boş erişim | Ana hat, derlemin kapsamadığı bir bölüm önerir | Yüksek skorlu adayı olmayan bölümleri atla ya da işaretle |
| Atıf kayması | LLM alıntılarda olmayan bir makaleye atıf yapar | Kaynakça LLM çıktısından değil, yapılandırılmış üst verilerden oluşturulur |
| Konu çöküşü | Tüm bölümler aynı 2 makaleyi getirir | Erişimde MMR (maximal marginal relevance) ile çeşitlendir |
| Maliyet patlaması | bölüm × candidate_k LLM çağrısı | Gömmeleri önbelleğe al; cross-encoder'ı yığınla; bölüm sayısını sınırla |
| Bayat indeks | Yeni bir konuda yeniden çalıştırmak eski bir koleksiyonu kullanır | Koleksiyon adı konu özetine göre anahtarlanır (bkz. `pipeline.py`) |

Ders: **entegrasyon testi, birim testlerinin toplamı değildir.** Her modül tek başına doğrudur; *birleşimin* kendi testlerine ve kendi korumalarına ihtiyacı vardır.

## 7. Alıştırmalar

1. **MMR çeşitlendirmesi.** Bölümlerin hepsi aynı makaleleri çekmesin diye `HierarchicalRAG.retrieve` yöntemine maximal marginal relevance ekleyin. Çeşitlilik/ilgililik $\lambda$ değerini ayarlayın.
2. **Ajan tabanlı sentez.** Tek seferlik bölüm sentezini, bölümün ortasında *ek* kanıt çekebilmesi için bir `search_corpus` aracı verilen Hafta 4 ReAct ajanıyla değiştirin.
3. **Öz-eleştiri.** Üretilen incelemenin üzerinden son bir geçiş olarak Hafta 5 Eleştirmen ajanını ekleyin; her atfı alıntılarla karşılaştırıp desteklenmeyen iddiaları işaretlesin.
4. **PDF alma.** `arxiv_ingest` modülünü (yalnızca özetleri değil) tam PDF'leri indirip ayrıştıracak şekilde `pdf` becerisiyle genişletin ve yeniden çalıştırın. Tam metin erişimi, yalnızca özetle erişime göre bölüm kalitesini ne kadar iyileştiriyor?

## 8. Ders değerlendirmesi

Altı hafta boyunca temel ilkelerden yola çıkarak şunları kurduk:

1. **MCP** — araçları ve kaynakları bir LLM'e sunan bir JSON-RPC protokol sunucusu.
2. **Hiyerarşik RAG** — parçalama stratejileri, gömme geometrisi ve sıfırdan bir HNSW indeksi.
3. **Yeniden sıralama** — Bi/Cross-Encoder ikiliği, sorgu dönüşümü ve HyDE.
4. **ReAct** — sağlam hata yönetimine sahip, saf Python bir akıl yürütme ve eyleme döngüsü.
5. **Çoklu ajan sistemleri** — sonlanması kanıtlanabilir, kendini düzelten bir Kodlayıcı–Yürütücü–Eleştirmen döngüsü.
6. **Bitirme projesi** — yukarıdakilerin tamamı, otonom bir araştırma asistanında birleştirilmiş hâlde.

Hiçbir yerde üst düzey bir ajan çatısı kullanılmadı. Bu depodaki her soyutlama sizin yazdığınız ve sahip olduğunuz bir soyutlamadır. Bir ajan kütüphanesini *çağırmak* ile onu *anlamak* arasındaki fark budur — ve bu deponun "sıfırdan" yaklaşımının anlatmak istediği de tam olarak budur.

➡ Doğal bir sonraki adım, bitirme projesini Hafta 1'deki MCP sunucusunun arkasına yerleştirmektir; böylece MCP uyumlu herhangi bir istemci araştırma asistanınızı bir araç olarak çağırabilir.